# ML Modeling

This notebook trains ML models and evaluates them for business impact.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/model_data.csv")
df.shape

In [ ]:
df["y"].value_counts(normalize=True)

In [ ]:
X = df.drop(columns=["y"])
y = df["y"]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_features

In [ ]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_features

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

logistic_model.fit(X_train, y_train)

logistic_probability = logistic_model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

print("ROC-AUC:", roc_auc_score(y_test, logistic_probability))
print("PR-AUC:", average_precision_score(y_test, logistic_probability))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1))
])

rf_model.fit(X_train, y_train)

rf_probability = rf_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, rf_probability))
print("PR-AUC:", average_precision_score(y_test, rf_probability))

In [ ]:
from xgboost import XGBClassifier

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42))
])

xgb_model.fit(X_train, y_train)

xgb_probability = xgb_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, xgb_probability))
print("PR-AUC:", average_precision_score(y_test, xgb_probability))

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "ROC_AUC": [
        roc_auc_score(y_test, logistic_probability),
        roc_auc_score(y_test, rf_probability),
        roc_auc_score(y_test, xgb_probability)
    ],
    "PR_AUC": [
        average_precision_score(y_test, logistic_probability),
        average_precision_score(y_test, rf_probability),
        average_precision_score(y_test, xgb_probability)
    ]
})

results.sort_values("PR_AUC", ascending=False)

In [ ]:
evaluation = X_test.copy()

evaluation["actual"] = y_test.values
evaluation["probability"] = xgb_probability

evaluation = evaluation.sort_values("probability", ascending=False)

evaluation.head()

In [ ]:
top_10_percent = evaluation.head(int(len(evaluation) * 0.10))

precision_at_10 = top_10_percent["actual"].mean()

print(f"Precision@10%: {precision_at_10:.2%}")

baseline_rate = y_test.mean()

print(f"Baseline conversion: {baseline_rate:.2%}")
print(f"Lift: {precision_at_10 / baseline_rate:.2f}x")

In [ ]:
evaluation["priority_rank"] = range(1, len(evaluation) + 1)

top_customers = evaluation[evaluation["priority_rank"] <= int(len(evaluation) * 0.10)].copy()
top_customers.head()

# Decision Engine

Generate predictions for the entire dataset.

In [ ]:
all_probability = xgb_model.predict_proba(X)[:, 1]

decision_df = X.copy()
decision_df["xgb_probability"] = all_probability
decision_df["actual"] = y.values

In [ ]:
from src.decision_engine.campaign_optimizer import (
    CampaignConfig,
    rank_customers
)

config = CampaignConfig(
    conversion_value=1000,
    contact_cost=20,
    budget=100000
)

ranked_customers = rank_customers(
    decision_df,
    probability_column="xgb_probability",
    config=config
)

ranked_customers[
    ["xgb_probability", "expected_value", "priority_rank"]
].head(20)

In [ ]:
profitable_customers = ranked_customers[
    ranked_customers["expected_value"] > 0
].copy()

profitable_customers[
    ["xgb_probability", "expected_value", "priority_rank"]
].head()

In [ ]:
from src.decision_engine.campaign_optimizer import select_customers

selected = select_customers(
    ranked_customers,
    config
)

estimated_conversions = selected["xgb_probability"].sum()
total_cost = selected["contact_cost"].sum()
expected_value = selected["expected_value"].sum()

print(f"Customers contacted: {len(selected):,}")
print(f"Expected conversions: {estimated_conversions:.2f}")
print(f"Campaign cost: ₹{total_cost:,.2f}")
print(f"Expected net value: ₹{expected_value:,.2f}")

In [ ]:
expected_roi = expected_value / total_cost
print(f"Expected ROI: {expected_roi:.2f}x")

In [ ]:
scenarios = {
    "Conservative": 500,
    "Base": 1000,
    "Optimistic": 2000
}

scenario_results = []

for name, value in scenarios.items():
    config = CampaignConfig(
        conversion_value=value,
        contact_cost=20,
        budget=100000
    )

    ranked = rank_customers(
        decision_df,
        "xgb_probability",
        config
    )

    selected = select_customers(
        ranked,
        config
    )

    scenario_results.append({
        "scenario": name,
        "conversion_value": value,
        "customers_targeted": len(selected),
        "expected_conversions": selected["xgb_probability"].sum(),
        "campaign_cost": selected["contact_cost"].sum(),
        "expected_net_value": selected["expected_value"].sum()
    })

scenario_results = pd.DataFrame(scenario_results)
scenario_results

In [ ]:
random_target = decision_df.sample(n=len(selected), random_state=42)

random_expected_conversions = random_target["actual"].mean() * len(random_target)
model_expected_conversions = selected["xgb_probability"].sum()

print("Random baseline expected conversions:", random_expected_conversions)
print("Model-targeted expected conversions:", model_expected_conversions)

In [ ]:
baseline_rate = y.mean()
targeted_rate = selected["actual"].mean()
lift = targeted_rate / baseline_rate

print(f"Baseline conversion rate: {baseline_rate:.2%}")
print(f"Targeted conversion rate: {targeted_rate:.2%}")
print(f"Lift: {lift:.2f}x")

In [ ]:
OUTPUT_PATH = "../data/processed/campaign_target_list.csv"

selected.to_csv(OUTPUT_PATH, index=False)

print(f"Saved campaign list to {OUTPUT_PATH}")